In [20]:
import cv2
import typing
import numpy as np

from mltu.inferenceModel import OnnxInferenceModel
from mltu.utils.text_utils import ctc_decoder, get_cer

In [21]:
class ImageToWordModel(OnnxInferenceModel):

    def __init__(self, char_list, *args, **kwargs):
        super().__init__(*args, **kwargs)

        self.char_list = char_list

        self.input_shapes = (32, 128, 3)

    def predict(self, image):

        image = cv2.resize(image, (128, 32))

        image = image.astype(np.float32) / 255.0

        image = np.expand_dims(image, axis=0)

        preds = self.model.run(
            None,
            {self.input_name: image}
        )[0]

        prediction_text = ctc_decoder(
            preds,
            self.char_list
        )[0]

        return prediction_text

In [22]:
if __name__ == "__main__":

    import os
    import cv2
    import numpy as np
    import pandas as pd

    from tqdm import tqdm
    from mltu.configs import BaseModelConfigs

    # -----------------------------
    # LOAD CONFIGS
    # -----------------------------
    configs = BaseModelConfigs.load(
        "../models/202211270035/configs.yaml"
    )

    # -----------------------------
    # FIX WRONG MODEL PATH
    # -----------------------------
    configs.model_path = "../models/202211270035/model.onnx"

    # -----------------------------
    # LOAD MODEL
    # -----------------------------
    model = ImageToWordModel(
        model_path=configs.model_path,
        char_list=configs.vocab
    )

    # -----------------------------
    # LOAD CSV
    # -----------------------------
    df = pd.read_csv(
        "../models/202211270035/val.csv"
    ).dropna().values.tolist()

    # -----------------------------
    # FILTER VALID FILES
    # -----------------------------
    valid_df = []

    for image_path, label in df:

        image_path = image_path.replace("\\", "/")

        # FIX DATASET PATH
        image_path = image_path.replace(
            "Datasets/90kDICT32px",
            "../datasets/mjsynth/mnt/ramdisk/max/90kDICT32px"
        )

        if os.path.exists(image_path):
            valid_df.append((image_path, label))

    print(f"\nTotal valid images: {len(valid_df)}")
    # -----------------------------
    # RUN INFERENCE
    # -----------------------------
    accum_cer = []

    for image_path, label in tqdm(valid_df[:20]):

        image = cv2.imread(image_path)

        if image is None:
            print(f"Failed to load: {image_path}")
            continue

        try:
            prediction_text = model.predict(image)

            cer = get_cer(prediction_text, label)

            print(
                f"\nImage: {image_path}\n"
                f"Label: {label}\n"
                f"Prediction: {prediction_text}\n"
                f"CER: {cer}\n"
            )

            accum_cer.append(cer)

        except Exception as e:
            print(f"Error processing {image_path}")
            print(e)

    # -----------------------------
    # FINAL METRIC
    # -----------------------------
    if len(accum_cer) > 0:
        print(f"\nAverage CER: {np.average(accum_cer)}")
    else:
        print("\nNo valid predictions were made.")


Total valid images: 802714


100%|█████████████████████████████████████████████████████████████████████████████████| 20/20 [00:00<00:00, 244.87it/s]


Image: ../datasets/mjsynth/mnt/ramdisk/max/90kDICT32px/2697/6/466_MONIKER_49537.jpg
Label: MONIKER
Prediction: n
CER: 1.0


Image: ../datasets/mjsynth/mnt/ramdisk/max/90kDICT32px/2697/6/465_Ecclesiastics_24500.jpg
Label: Ecclesiastics
Prediction: n
CER: 1.0


Image: ../datasets/mjsynth/mnt/ramdisk/max/90kDICT32px/2697/6/464_FIRESTORM_29099.jpg
Label: FIRESTORM
Prediction: n
CER: 1.0


Image: ../datasets/mjsynth/mnt/ramdisk/max/90kDICT32px/2697/6/463_Psi_60982.jpg
Label: Psi
Prediction: n
CER: 1.0


Image: ../datasets/mjsynth/mnt/ramdisk/max/90kDICT32px/2697/6/462_Repurchases_64997.jpg
Label: Repurchases
Prediction: n
CER: 1.0


Image: ../datasets/mjsynth/mnt/ramdisk/max/90kDICT32px/2697/6/461_PIGTAIL_57575.jpg
Label: PIGTAIL
Prediction: n
CER: 1.0


Image: ../datasets/mjsynth/mnt/ramdisk/max/90kDICT32px/2697/6/460_landladies_43270.jpg
Label: landladies
Prediction: n
CER: 0.9


Image: ../datasets/mjsynth/mnt/ramdisk/max/90kDICT32px/2697/6/459_Silliest_70946.jpg
Label: Silliest
Predicti